# `myls` — a directory-listing utility in C, built directly on Linux system calls

| | |
|---|---|
| **Course / lab** | 22CS3C05L — Computer Science III Lab |
| **Topic** | Directory and file command manipulation |
| **Simulates** | Technical practical round |
| **Student** | *your name* · *roll number* |
| **Executed on** | 20 November 2026, Linux 6.x, Ubuntu 24.04 |
| **Tools** | GCC 13.3 · GNU coreutils `ls` 9.4 (reference implementation) · Claude (design review, edge-case ideas) |
| **Proof submitted** | This notebook: complete C source + live execution logs |

**Contents** — 1 Aim · 2 Theory · 3 Source code · 4 Build · 5 Demonstration · 6 Permission-flag verification ·
7 Error handling & access control · 8 Automated test suite · 9 Memory-safety & mutation testing · 10 Results ·
11 Bugs found during testing · 12 Limitations · 13 Viva questions · 14 AI-assistance note

> Every output below was produced by actually compiling and running the code in this notebook; nothing is hand-typed.

## 1. Aim

Implement a command-line utility in C that lists directory contents like the standard `ls`, **using system calls**
to read directories and file metadata. Test it on varied directory structures and **verify the file-permission flags**.

**Objectives**

- [x] Read directories with `open` + `getdents64`, not the buffered `opendir/readdir` wrappers
- [x] Fetch metadata with `lstat`/`stat`, resolve links with `readlink`
- [x] Support `-a -d -l -R -i -1 -h -t -S -r -F` (+ `-O`, an octal-permission extension used for verification)
- [x] Decode permission bits, including setuid / setgid / sticky (`s S t T`)
- [x] Test on flat, hidden, nested, empty, huge, symlinked and special-file structures
- [x] Prove correctness against GNU `ls` and against `stat(1)` for **all 4096** permission patterns
- [x] Verify access-control behaviour as an unprivileged user (root bypasses permission bits)

**Success criteria** — output byte-identical to GNU `ls` for every supported option, identical diagnostics and exit
status, zero sanitizer findings, and a test-suite that demonstrably fails when the code is deliberately broken.

## 2. Theory

### 2.1 What `ls` actually does
Three kernel services, plus formatting: **(1)** enumerate the names in a directory, **(2)** fetch metadata for each
name, **(3)** read symbolic-link targets.

| System call | Purpose | Used in |
|---|---|---|
| `open(path, O_RDONLY\|O_DIRECTORY)` | file descriptor for a directory (`ENOTDIR` on files, `EACCES` without read permission) | `list_directory` |
| `getdents64(fd, buf, n)` | fills `buf` with variable-length `linux_dirent64` records (inode, type hint, name). `readdir()` is only a buffered wrapper around this | `list_directory` |
| `lstat(path, &st)` | inode metadata **without** following symlinks | `fill_entry`, `main` |
| `stat(path, &st)` | same, but follows symlinks | symlink operands, `-F` link targets |
| `readlink(path, buf, n)` | text of a symlink's target | `fill_entry` |
| `ioctl(1, TIOCGWINSZ, &ws)` | terminal width for column layout | `term_width` |
| `close(fd)` | release the descriptor | `list_directory` |

`glibc` has no wrapper for `getdents64`, so it is called through `syscall(SYS_getdents64, …)`. (`getpwuid`/`getgrgid`
for user names and `strftime` for dates are library helpers, not system calls.)

### 2.2 Anatomy of `st_mode`
```
bit   15 14 13 12 | 11    10    9      | 8  7  6 | 5  4  3 | 2  1  0
      file type   | SUID  SGID  STICKY | u: r w x| g: r w x| o: r w x
```
File-type nibble → character: regular `-` · directory `d` · symlink `l` · char device `c` · block device `b` · FIFO `p` · socket `s`.

The three special bits are *displayed inside the execute slot* of their class:

| bit (octal) | slot | execute also set → | execute **not** set → |
|---|---|---|---|
| setuid `04000` | owner `x` | `s` | `S` |
| setgid `02000` | group `x` | `s` | `S` |
| sticky `01000` | other `x` | `t` | `T` |

Example: `4755` → `-rwsr-xr-x` · `1777` on a directory → `drwxrwxrwt` (like `/tmp`) · `4644` → `-rwSr--r--`.

### 2.3 Algorithm
```
for each operand: lstat() it            -> files vs. directories  (symlink-to-dir is followed unless -l / -d)
print files (sorted), then for each directory (sorted):
    fd = open(dir, O_DIRECTORY)
    loop: n = getdents64(fd, buf)       -> parse records until n == 0   (handles directories of any size)
          skip dotfiles unless -a
          lstat(dir/name) only if -l -t -S -F need it   (so a readable-but-unsearchable dir still lists names)
    qsort(entries)  ->  by name | -t mtime | -S size, optionally reversed (-r)
    print: short, columns (tty) or long (widths computed first so columns align)
    if -R: recurse into every real directory (lstat => symlinks never followed => no cycles)
```
Cost: O(n log n) time, O(n) memory per directory.

### 2.4 Exit status (same convention as GNU `ls`)
`0` success · `1` minor problem (e.g. unreadable sub-directory found while recursing) · `2` serious problem
(a command-line operand could not be accessed, or bad option).

## 3. Source code

### 3.1 Environment
Force the C locale (byte-order sorting, identical to GNU `ls` under `LC_ALL=C`) and UTC for reproducible timestamps.

In [1]:
%env LC_ALL=C
%env TZ=UTC

env: LC_ALL=C
env: TZ=UTC


In [2]:
%%bash
gcc --version | head -1
ls --version | head -1
uname -sr
echo "uid=$(id -u)  ($(id -un))"

gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
ls (GNU coreutils) 9.4
Linux 6.18.44-fc-v726
uid=0  (root)


### 3.2 `myls.c` — the complete program

In [3]:
%%writefile myls.c
/*
 * myls.c - a small `ls` clone built directly on Linux system calls.
 *
 * Course  : 22CS3C05L Computer Science III Lab
 * Topic   : Directory and file command manipulation
 *
 * System calls used directly
 *   open(2)          open a directory (O_DIRECTORY)
 *   getdents64(2)    read raw directory entries (via syscall(2); glibc has no wrapper)
 *   lstat(2)/stat(2) fetch inode metadata (type, mode, links, uid/gid, size, times)
 *   readlink(2)      resolve symbolic link targets
 *   ioctl(2)         TIOCGWINSZ - terminal width for column output
 *   close(2), isatty(3)
 * Library helpers (not system calls, used only for formatting)
 *   getpwuid/getgrgid (name lookup), localtime/strftime, qsort, printf
 *
 * Supported options
 *   -a  show hidden entries (including . and ..)
 *   -d  list directories themselves, not their contents (ls -ld dir)
 *   -l  long listing (type+permissions, links, owner, group, size, mtime)
 *   -i  print inode number
 *   -R  recurse into subdirectories
 *   -1  one entry per line (default when stdout is not a terminal)
 *   -h  human readable sizes (with -l)
 *   -t  sort by modification time, newest first
 *   -S  sort by size, largest first
 *   -r  reverse the sort order
 *   -F  append indicator (/ * @ | =) to names
 *   -O  print the permission bits in octal (e.g. 4755) before the symbolic form
 *       (extension, makes permission verification easy)
 *
 * Exit status: 0 ok, 1 minor problem (e.g. unreadable sub-directory),
 *              2 serious problem (e.g. command-line argument not found)
 */
#define _GNU_SOURCE
#include <dirent.h>   /* only for the DT_* constants; no opendir/readdir used */
#include <errno.h>
#include <fcntl.h>
#include <grp.h>
#include <pwd.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/ioctl.h>
#include <sys/stat.h>
#include <sys/sysmacros.h>
#include <sys/syscall.h>
#include <sys/types.h>
#include <time.h>
#include <unistd.h>

/* ---------- raw kernel directory record (see getdents64(2)) ---------- */
struct linux_dirent64 {
    uint64_t       d_ino;
    int64_t        d_off;
    unsigned short d_reclen;
    unsigned char  d_type;
    char           d_name[];
};

/* ---------- options ---------- */
static int opt_a, opt_d, opt_l, opt_i, opt_R, opt_1, opt_h, opt_t, opt_S, opt_r, opt_F, opt_O;
static int exit_status = 0;
static const char *prog = "myls";

/* ---------- one directory entry plus its metadata ---------- */
typedef struct {
    char       *name;      /* name as shown                         */
    char       *path;      /* path used for lstat/readlink          */
    struct stat st;
    int         have_st;   /* 0 if lstat failed or was not needed   */
    uint64_t    ino;       /* inode number from the dirent          */
    unsigned char dtype;   /* d_type hint from getdents64           */
    char       *link;      /* symlink target (only with -l)         */
} entry;

/* ================= helpers ================= */

static void *xmalloc(size_t n)
{
    void *p = malloc(n ? n : 1);
    if (!p) { perror(prog); exit(2); }
    return p;
}

static char *xstrdup(const char *s)
{
    char *p = xmalloc(strlen(s) + 1);
    strcpy(p, s);
    return p;
}

/* Print an error without letting it interleave badly with buffered stdout. */
static void report(const char *what, const char *path)
{
    int e = errno;
    fflush(stdout);
    fprintf(stderr, "%s: %s '%s': %s\n", prog, what, path, strerror(e));
}

/* "-rwsr-xr-t" style permission string. out must hold 11 bytes. */
void mode_string(mode_t m, char *s)
{
    char t;
    switch (m & S_IFMT) {
    case S_IFREG:  t = '-'; break;
    case S_IFDIR:  t = 'd'; break;
    case S_IFLNK:  t = 'l'; break;
    case S_IFCHR:  t = 'c'; break;
    case S_IFBLK:  t = 'b'; break;
    case S_IFIFO:  t = 'p'; break;
    case S_IFSOCK: t = 's'; break;
    default:       t = '?'; break;
    }
    s[0] = t;
    s[1] = (m & S_IRUSR) ? 'r' : '-';
    s[2] = (m & S_IWUSR) ? 'w' : '-';
    s[3] = (m & S_ISUID) ? ((m & S_IXUSR) ? 's' : 'S') : ((m & S_IXUSR) ? 'x' : '-');
    s[4] = (m & S_IRGRP) ? 'r' : '-';
    s[5] = (m & S_IWGRP) ? 'w' : '-';
    s[6] = (m & S_ISGID) ? ((m & S_IXGRP) ? 's' : 'S') : ((m & S_IXGRP) ? 'x' : '-');
    s[7] = (m & S_IROTH) ? 'r' : '-';
    s[8] = (m & S_IWOTH) ? 'w' : '-';
    s[9] = (m & S_ISVTX) ? ((m & S_IXOTH) ? 't' : 'T') : ((m & S_IXOTH) ? 'x' : '-');
    s[10] = '\0';
}

static void human_size(unsigned long long sz, char *out, size_t n)
{
    static const char suf[] = "KMGTPE";
    if (sz < 1024) { snprintf(out, n, "%llu", sz); return; }
    double v = (double)sz;
    int i = -1;
    while (v >= 1024.0 && i < 5) { v /= 1024.0; i++; }
    if (v < 10.0) {                         /* one decimal, rounded up like GNU ls */
        unsigned long t = (unsigned long)(v * 10.0);
        if ((double)t < v * 10.0) t++;
        if (t >= 100) snprintf(out, n, "10%c", suf[i]);
        else          snprintf(out, n, "%lu.%lu%c", t / 10, t % 10, suf[i]);
    } else {
        unsigned long t = (unsigned long)v;
        if ((double)t < v) t++;
        snprintf(out, n, "%lu%c", t, suf[i]);
    }
}

static char classify_mode(mode_t m)
{
    if (S_ISDIR(m))  return '/';
    if (S_ISLNK(m))  return '@';
    if (S_ISFIFO(m)) return '|';
    if (S_ISSOCK(m)) return '=';
    if (S_ISREG(m) && (m & (S_IXUSR | S_IXGRP | S_IXOTH))) return '*';
    return 0;
}

static char classify_char(const entry *e)
{
    return e->have_st ? classify_mode(e->st.st_mode) : 0;
}

/* ================= sorting ================= */

static int cmp_entry(const void *pa, const void *pb)
{
    const entry *a = pa, *b = pb;
    int r = 0;
    if (opt_S && a->have_st && b->have_st) {
        if (a->st.st_size != b->st.st_size)
            r = (a->st.st_size < b->st.st_size) ? 1 : -1;      /* big first */
    } else if (opt_t && a->have_st && b->have_st) {
        if (a->st.st_mtim.tv_sec != b->st.st_mtim.tv_sec)
            r = (a->st.st_mtim.tv_sec < b->st.st_mtim.tv_sec) ? 1 : -1; /* new first */
        else if (a->st.st_mtim.tv_nsec != b->st.st_mtim.tv_nsec)
            r = (a->st.st_mtim.tv_nsec < b->st.st_mtim.tv_nsec) ? 1 : -1;
    }
    if (r == 0) r = strcmp(a->name, b->name);
    return opt_r ? -r : r;
}

/* ================= printing ================= */

static int term_width(void)
{
    struct winsize ws;
    if (ioctl(STDOUT_FILENO, TIOCGWINSZ, &ws) == 0 && ws.ws_col > 0)
        return ws.ws_col;
    return 80;
}

static int ndigits(unsigned long long v)
{
    int d = 1;
    while (v >= 10) { v /= 10; d++; }
    return d;
}

static void print_name(const entry *e)
{
    fputs(e->name, stdout);
    if (opt_F) { char c = classify_char(e); if (c) putchar(c); }
}

/* Name lookups go through NSS and are slow; consecutive entries usually share owner/group, so cache the last one. */
static const char *user_name(uid_t uid, char *buf)
{
    static uid_t last = (uid_t)-1; static char name[32];
    if (uid != last) {
        struct passwd *pw = getpwuid(uid);
        if (pw) snprintf(name, sizeof name, "%s", pw->pw_name); else snprintf(name, sizeof name, "%u", uid);
        last = uid;
    }
    snprintf(buf, 32, "%s", name);
    return buf;
}

static const char *group_name(gid_t gid, char *buf)
{
    static gid_t last = (gid_t)-1; static char name[32];
    if (gid != last) {
        struct group *gr = getgrgid(gid);
        if (gr) snprintf(name, sizeof name, "%s", gr->gr_name); else snprintf(name, sizeof name, "%u", gid);
        last = gid;
    }
    snprintf(buf, 32, "%s", name);
    return buf;
}

/* Format owner, group and size columns for one entry. */
static void fmt_fields(const struct stat *s, char *own, char *grp, char *sz)
{
    user_name(s->st_uid, own);
    group_name(s->st_gid, grp);
    if (S_ISCHR(s->st_mode) || S_ISBLK(s->st_mode))
        snprintf(sz, 40, "%u, %u", major(s->st_rdev), minor(s->st_rdev));
    else if (opt_h)
        human_size((unsigned long long)s->st_size, sz, 40);
    else
        snprintf(sz, 40, "%lld", (long long)s->st_size);
}

struct widths { int ino, nl, own, grp, sz, maj, min; };

static void widen(struct widths *w, const entry *e)
{
    if (!e->have_st) return;
    char own[32], grp[32], sz[40];
    fmt_fields(&e->st, own, grp, sz);
    if (ndigits(e->st.st_ino)   > w->ino) w->ino = ndigits(e->st.st_ino);
    if (ndigits(e->st.st_nlink) > w->nl)  w->nl  = ndigits(e->st.st_nlink);
    if ((int)strlen(own) > w->own) w->own = (int)strlen(own);
    if ((int)strlen(grp) > w->grp) w->grp = (int)strlen(grp);
    if (S_ISCHR(e->st.st_mode) || S_ISBLK(e->st.st_mode)) {
        if (ndigits(major(e->st.st_rdev)) > w->maj) w->maj = ndigits(major(e->st.st_rdev));
        if (ndigits(minor(e->st.st_rdev)) > w->min) w->min = ndigits(minor(e->st.st_rdev));
    } else if ((int)strlen(sz) > w->sz) {
        w->sz = (int)strlen(sz);
    }
}

/*
 * Long listing. `extra` are additional entries (the directory operands) that
 * GNU ls also counts when it computes column widths for the file operands.
 */
static void print_long(entry *v, size_t n, int show_total, const entry *extra, size_t nextra)
{
    struct widths w = {0, 0, 0, 0, 0, 0, 0};
    unsigned long long blocks = 0;
    for (size_t i = 0; i < n; i++)      { widen(&w, &v[i]); if (v[i].have_st) blocks += (unsigned long long)v[i].st.st_blocks; }
    for (size_t i = 0; i < nextra; i++) widen(&w, &extra[i]);
    if (w.maj || w.min) {                       /* device files present: "major, minor" must fit the size column */
        if (w.maj + w.min + 2 > w.sz) w.sz = w.maj + w.min + 2;
        else                          w.maj = w.sz - w.min - 2;
    }

    if (show_total) {
        unsigned long long kb = (blocks + 1) / 2;      /* 512-byte units -> 1K, rounded up */
        if (opt_h) { char b[40]; human_size(kb * 1024ULL, b, sizeof b); printf("total %s\n", b); }
        else       printf("total %llu\n", kb);
    }

    time_t now = time(NULL);
    for (size_t i = 0; i < n; i++) {
        entry *e = &v[i];
        if (!e->have_st) {
            printf("?????????? ? ? ? ? ? %s\n", e->name);
            continue;
        }
        const struct stat *s = &e->st;
        char ms[11], tbuf[32], own[32], grp[32], sz[40];
        mode_string(s->st_mode, ms);
        fmt_fields(s, own, grp, sz);

        struct tm tm;
        localtime_r(&s->st_mtim.tv_sec, &tm);
        long long age = (long long)now - (long long)s->st_mtim.tv_sec;
        /* GNU ls: files older than ~6 months (or in the future) show the year */
        if (age > 15778476LL || age < 0) strftime(tbuf, sizeof tbuf, "%b %e  %Y", &tm);
        else                              strftime(tbuf, sizeof tbuf, "%b %e %H:%M", &tm);

        if (opt_i) printf("%*llu ", w.ino, (unsigned long long)s->st_ino);
        if (opt_O) printf("%04o ", (unsigned)(s->st_mode & 07777));
        if (S_ISCHR(s->st_mode) || S_ISBLK(s->st_mode))
            snprintf(sz, sizeof sz, "%*u, %*u", w.maj, major(s->st_rdev), w.min, minor(s->st_rdev));
        printf("%s %*llu %-*s %-*s %*s %s ", ms,
               w.nl, (unsigned long long)s->st_nlink,
               w.own, own, w.grp, grp, w.sz, sz, tbuf);

        if (S_ISLNK(s->st_mode)) {
            fputs(e->name, stdout);                      /* no indicator on the link name itself */
            if (e->link) {
                printf(" -> %s", e->link);
                if (opt_F) {                             /* GNU ls classifies the link *target* */
                    struct stat ts;
                    if (stat(e->path, &ts) == 0) { char c = classify_mode(ts.st_mode); if (c) putchar(c); }
                }
            }
        } else {
            print_name(e);
        }
        putchar('\n');
    }
}

static void print_short(entry *v, size_t n)
{
    int tty = isatty(STDOUT_FILENO);
    if (opt_1 || !tty || n == 0) {
        for (size_t i = 0; i < n; i++) {
            if (opt_i) printf("%llu ", (unsigned long long)v[i].ino);
            print_name(&v[i]);
            putchar('\n');
        }
        return;
    }
    /* column layout, filled top-to-bottom like ls */
    size_t maxlen = 0;
    for (size_t i = 0; i < n; i++) {
        size_t l = strlen(v[i].name) + (opt_F && classify_char(&v[i]) ? 1 : 0);
        if (opt_i) l += (size_t)ndigits(v[i].ino) + 1;
        if (l > maxlen) maxlen = l;
    }
    size_t colw = maxlen + 2;
    size_t cols = (size_t)term_width() / colw;
    if (cols < 1) cols = 1;
    size_t rows = (n + cols - 1) / cols;
    for (size_t r = 0; r < rows; r++) {
        for (size_t c = 0; c < cols; c++) {
            size_t idx = c * rows + r;
            if (idx >= n) break;
            size_t l = strlen(v[idx].name) + (opt_F && classify_char(&v[idx]) ? 1 : 0);
            if (opt_i) { printf("%llu ", (unsigned long long)v[idx].ino);
                         l += (size_t)ndigits(v[idx].ino) + 1; }
            print_name(&v[idx]);
            if (idx + rows < n) for (size_t k = l; k < colw; k++) putchar(' ');
        }
        putchar('\n');
    }
}

/* ================= entry management ================= */

/* Record a failure: serious (2) for command-line operands, minor (1) otherwise. */
static void note_failure(int cmdline)
{
    if (cmdline) exit_status = 2;
    else if (!exit_status) exit_status = 1;
}

static void fill_entry(entry *e, const char *name, const char *path, int follow, int need_stat, int cmdline)
{
    e->name = xstrdup(name);
    e->path = xstrdup(path);
    e->link = NULL;
    e->have_st = 0;
    e->ino = 0;
    e->dtype = DT_UNKNOWN;
    if (!need_stat) return;               /* like GNU ls: no stat() unless required */
    e->have_st = (follow ? stat(path, &e->st) : lstat(path, &e->st)) == 0;
    if (!e->have_st) { report("cannot access", path); note_failure(cmdline); return; }
    e->ino = e->st.st_ino;
    if (opt_l && S_ISLNK(e->st.st_mode)) {
        char buf[4096];
        ssize_t k = readlink(path, buf, sizeof buf - 1);
        if (k >= 0) { buf[k] = '\0'; e->link = xstrdup(buf); }
        else { report("cannot read symbolic link", path); note_failure(cmdline); }
    }
}

static void free_entries(entry *v, size_t n)
{
    for (size_t i = 0; i < n; i++) { free(v[i].name); free(v[i].path); free(v[i].link); }
    free(v);
}

/* ================= directory listing ================= */

static void list_directory(const char *dir, int print_header, int *first, int cmdline)
{
    int fd = open(dir, O_RDONLY | O_DIRECTORY | O_CLOEXEC);
    if (fd < 0) {
        report("cannot open directory", dir);
        if (cmdline) exit_status = 2;                         /* serious: user asked for it */
        else if (!exit_status) exit_status = 1;               /* minor: found while recursing */
        return;
    }

    int need_stat = opt_l || opt_t || opt_S || opt_F;
    entry  *v = NULL;
    size_t  n = 0, cap = 0;
    char    buf[32768];
    long    nread;

    while ((nread = syscall(SYS_getdents64, fd, buf, sizeof buf)) > 0) {
        for (long off = 0; off < nread; ) {
            struct linux_dirent64 *d = (struct linux_dirent64 *)(buf + off);
            off += d->d_reclen;
            if (!opt_a && d->d_name[0] == '.') continue;      /* hide dotfiles */

            if (n == cap) { cap = cap ? cap * 2 : 64; v = realloc(v, cap * sizeof *v); if (!v) { perror(prog); exit(2); } }
            size_t plen = strlen(dir) + strlen(d->d_name) + 2;
            char *path = xmalloc(plen);
            if (dir[0] != '\0' && dir[strlen(dir) - 1] == '/') snprintf(path, plen, "%s%s", dir, d->d_name);
            else                                               snprintf(path, plen, "%s/%s", dir, d->d_name);
            fill_entry(&v[n], d->d_name, path, 0, need_stat, 0);
            if (!v[n].ino) v[n].ino = d->d_ino;
            v[n].dtype = d->d_type;
            free(path);
            n++;
        }
    }
    if (nread < 0) { report("error reading directory", dir); exit_status = exit_status ? exit_status : 1; }
    close(fd);

    if (n > 1) qsort(v, n, sizeof *v, cmp_entry);

    if (print_header) { if (!*first) putchar('\n'); printf("%s:\n", dir); }
    *first = 0;

    if (opt_l) print_long(v, n, 1, NULL, 0);
    else       print_short(v, n);

    if (opt_R) {
        for (size_t i = 0; i < n; i++) {
            int isdir = v[i].have_st ? S_ISDIR(v[i].st.st_mode) : v[i].dtype == DT_DIR;
            if (v[i].dtype == DT_UNKNOWN && !v[i].have_st) {          /* filesystem gave no type hint */
                struct stat tmp;
                isdir = lstat(v[i].path, &tmp) == 0 && S_ISDIR(tmp.st_mode);
            }
            if (!isdir) continue;
            if (!strcmp(v[i].name, ".") || !strcmp(v[i].name, "..")) continue;
            list_directory(v[i].path, 1, first, 0);
        }
    }
    free_entries(v, n);
}

/* ================= main ================= */

int main(int argc, char **argv)
{
    int c;
    prog = "myls";
    opterr = 0;                                   /* we print our own option diagnostics */
    while ((c = getopt(argc, argv, "adlRiShtr1FO")) != -1) {
        switch (c) {
        case 'a': opt_a = 1; break;
        case 'd': opt_d = 1; break;
        case 'l': opt_l = 1; break;
        case 'R': opt_R = 1; break;
        case 'i': opt_i = 1; break;
        case 'S': opt_S = 1; break;
        case 'h': opt_h = 1; break;
        case 't': opt_t = 1; break;
        case 'r': opt_r = 1; break;
        case '1': opt_1 = 1; break;
        case 'F': opt_F = 1; break;
        case 'O': opt_O = 1; opt_l = 1; break;
        default:
            fprintf(stderr, "%s: invalid option -- '%c'\nusage: %s [-adlRiShtr1FO] [file ...]\n", prog, optopt ? optopt : '?', prog);
            return 2;
        }
    }

    int nargs = argc - optind;
    char **args = argv + optind;
    static char *dot[] = { ".", NULL };
    if (nargs == 0) { args = dot; nargs = 1; }

    /* Split command-line operands into files and directories (like GNU ls). */
    entry *files = xmalloc((size_t)nargs * sizeof *files);
    entry *dirs  = xmalloc((size_t)nargs * sizeof *dirs);
    size_t nf = 0, nd = 0;

    for (int i = 0; i < nargs; i++) {
        struct stat st;
        if (lstat(args[i], &st) != 0) {
            report("cannot access", args[i]);
            exit_status = 2;
            continue;
        }
        /* Without -l, a symlink to a directory is followed (as in GNU ls). */
        int follow = S_ISLNK(st.st_mode) && !opt_l && !opt_d && stat(args[i], &st) == 0 && S_ISDIR(st.st_mode);
        entry tmp;
        fill_entry(&tmp, args[i], args[i], follow, 1, 1);
        if (!tmp.have_st) continue;
        if (S_ISDIR(tmp.st.st_mode) && !opt_d) dirs[nd++] = tmp;
        else                         files[nf++] = tmp;
    }

    if (nf > 1) qsort(files, nf, sizeof *files, cmp_entry);
    if (nd > 1) qsort(dirs,  nd, sizeof *dirs,  cmp_entry);

    int first = 1;
    if (nf) {
        if (opt_l) print_long(files, nf, 0, dirs, nd);
        else       print_short(files, nf);
        first = 0;
    }
    int header = (nargs > 1) || opt_R;
    for (size_t i = 0; i < nd; i++)
        list_directory(dirs[i].path, header, &first, 1);

    free_entries(files, nf);
    free_entries(dirs, nd);
    fflush(stdout);
    return exit_status;
}


Writing myls.c


## 4. Build
Strict warnings (`-Wall -Wextra -Wpedantic`) — the build must be warning-free.

In [4]:
%%bash
gcc -std=gnu11 -Wall -Wextra -Wpedantic -O2 -o myls myls.c && echo "BUILD OK: $(wc -l < myls.c) lines of C, binary $(stat -c %s myls) bytes"

BUILD OK: 529 lines of C, binary 27040 bytes


**Proof that real system calls are used**: the dynamic symbols the binary imports. `open`, `syscall` (→ `getdents64`), `lstat`, `stat`, `readlink`, `ioctl` are there; `opendir`/`readdir` are *absent*.

In [5]:
%%bash
nm -D --undefined-only myls | awk '{print $2}' | sed 's/@.*//' | grep -E '^(syscall|lstat|stat|open|readlink|ioctl|close|isatty|opendir|readdir)' | sort | tr '\n' ' '; echo
echo "opendir/readdir imported: $(nm -D --undefined-only myls | grep -cE 'opendir|readdir')"

close ioctl isatty lstat open readlink stat syscall 
opendir/readdir imported: 0


## 5. Demonstration on a sample project tree

In [6]:
%%bash
rm -rf demo && mkdir -p demo/src/utils demo/docs demo/empty demo/.git
printf 'int main(void){return 0;}\n' > demo/src/main.c
printf '/* helpers */\n'             > demo/src/utils/helper.c
printf '# Demo project\n'            > demo/README.md
printf '#!/bin/sh\necho hi\n'        > demo/run.sh && chmod 755 demo/run.sh
echo 'API_KEY=secret'                > demo/.env   && chmod 600 demo/.env
head -c 2048    /dev/zero > demo/docs/manual.pdf
head -c 1500000 /dev/zero > demo/docs/video.bin
ln -s README.md demo/readme.lnk
ln -s src       demo/srclink
touch -d '2024-01-15 09:30' demo/docs/manual.pdf
touch -d '2 hours ago'      demo/docs/video.bin
echo "Structure created (find | sort):"; find demo | sort


Structure created (find | sort):
demo
demo/.env
demo/.git
demo/README.md
demo/docs
demo/docs/manual.pdf
demo/docs/video.bin
demo/empty
demo/readme.lnk
demo/run.sh
demo/src
demo/src/main.c
demo/src/utils
demo/src/utils/helper.c
demo/srclink


In [7]:
%%bash
echo '$ ./myls demo';            ./myls demo
echo; echo '$ ./myls -a demo';   ./myls -a demo
echo; echo '$ ./myls -F demo';   ./myls -F demo
echo; echo '$ ./myls -i demo/src'; ./myls -i demo/src


$ ./myls demo
README.md
docs
empty
readme.lnk
run.sh
src
srclink

$ ./myls -a demo
.
..
.env
.git
README.md
docs
empty
readme.lnk
run.sh
src
srclink

$ ./myls -F demo
README.md
docs/
empty/
readme.lnk@
run.sh*
src/
srclink@

$ ./myls -i demo/src
622604 main.c
622600 utils


In [8]:
%%bash
echo '$ ./myls -la demo'; ./myls -la demo
echo; echo '$ ./myls -lh demo/docs'; ./myls -lh demo/docs
echo; echo '--- compare with GNU ls ---'
diff <(ls -la demo)      <(./myls -la demo)      && echo 'myls -la  demo       : IDENTICAL to GNU ls'
diff <(ls -lh demo/docs) <(./myls -lh demo/docs) && echo 'myls -lh  demo/docs  : IDENTICAL to GNU ls'


$ ./myls -la demo
total 36
drwxr-xr-x 6 root root 4096 Nov 20 12:27 .
drwxr-xr-x 3 root root 4096 Nov 20 12:27 ..
-rw------- 1 root root   15 Nov 20 12:27 .env
drwxr-xr-x 2 root root 4096 Nov 20 12:27 .git
-rw-r--r-- 1 root root   15 Nov 20 12:27 README.md
drwxr-xr-x 2 root root 4096 Nov 20 12:27 docs
drwxr-xr-x 2 root root 4096 Nov 20 12:27 empty
lrwxrwxrwx 1 root root    9 Nov 20 12:27 readme.lnk -> README.md
-rwxr-xr-x 1 root root   18 Nov 20 12:27 run.sh
drwxr-xr-x 3 root root 4096 Nov 20 12:27 src
lrwxrwxrwx 1 root root    3 Nov 20 12:27 srclink -> src

$ ./myls -lh demo/docs
total 1.5M
-rw-r--r-- 1 root root 2.0K Jan 15  2024 manual.pdf
-rw-r--r-- 1 root root 1.5M Nov 20 10:27 video.bin

--- compare with GNU ls ---
myls -la  demo       : IDENTICAL to GNU ls
myls -lh  demo/docs  : IDENTICAL to GNU ls


In [9]:
%%bash
echo '$ ./myls -R demo'; ./myls -R demo
echo; echo '$ ./myls -lS demo/docs    (largest first)'; ./myls -lS demo/docs
echo; echo '$ ./myls -ltr demo/docs   (oldest first)';   ./myls -ltr demo/docs
echo; echo '$ ./myls -d demo demo/src demo/README.md'; ./myls -d demo demo/src demo/README.md


$ ./myls -R demo
demo:
README.md
docs
empty
readme.lnk
run.sh
src
srclink

demo/docs:
manual.pdf
video.bin

demo/empty:

demo/src:
main.c
utils

demo/src/utils:
helper.c

$ ./myls -lS demo/docs    (largest first)
total 1472
-rw-r--r-- 1 root root 1500000 Nov 20 10:27 video.bin
-rw-r--r-- 1 root root    2048 Jan 15  2024 manual.pdf

$ ./myls -ltr demo/docs   (oldest first)
total 1472
-rw-r--r-- 1 root root    2048 Jan 15  2024 manual.pdf
-rw-r--r-- 1 root root 1500000 Nov 20 10:27 video.bin

$ ./myls -d demo demo/src demo/README.md
demo
demo/README.md
demo/src


In [10]:
%%bash
ln demo/README.md demo/README.hard
echo '$ ./myls -li demo | grep -E "README|readme|srclink"'
./myls -li demo | grep -E 'README|readme|srclink'
echo
echo '  -> README.md and README.hard share one inode and show link count 2 (hard link).'
echo '  -> readme.lnk / srclink are separate inodes of type l (symbolic links) showing "-> target".'
echo; echo '$ ./myls demo/srclink      (symlink to a directory is followed without -l)'; ./myls demo/srclink
echo; echo '$ ./myls -ld demo/srclink  (-l / -d do not follow it)';                        ./myls -ld demo/srclink


$ ./myls -li demo | grep -E "README|readme|srclink"
622606 -rw-r--r-- 2 root root   15 Nov 20 12:27 README.hard
622606 -rw-r--r-- 2 root root   15 Nov 20 12:27 README.md
622611 lrwxrwxrwx 1 root root    9 Nov 20 12:27 readme.lnk -> README.md
622612 lrwxrwxrwx 1 root root    3 Nov 20 12:27 srclink -> src

  -> README.md and README.hard share one inode and show link count 2 (hard link).
  -> readme.lnk / srclink are separate inodes of type l (symbolic links) showing "-> target".

$ ./myls demo/srclink      (symlink to a directory is followed without -l)
main.c
utils

$ ./myls -ld demo/srclink  (-l / -d do not follow it)
lrwxrwxrwx 1 root root 3 Nov 20 12:27 demo/srclink -> src


## 6. Permission-flag verification

### 6.1 Hand-built fixtures, decoded by `myls -lO` and checked against `stat(1)`
`-O` is a small extension that prints the octal mode (`0644`) before the symbolic string, so both views can be
checked at once. `stat` is the independent reference.

In [11]:
%%bash
rm -rf perms_lab && mkdir perms_lab && cd perms_lab
for m in 000 400 600 644 755 777 4755 2755 4644 1644 7777; do touch f_$m; chmod $m f_$m; done
mkdir d_755 d_700 d_1777 d_2775; chmod 755 d_755; chmod 700 d_700; chmod 1777 d_1777; chmod 2775 d_2775
cd ..
echo '$ ./myls -lO perms_lab'
./myls -lO perms_lab
echo
exp=$(cd perms_lab && for f in $(ls -A); do printf '%04d %s %s\n' "$(stat -c %a $f)" "$(stat -c %A $f)" "$f"; done)
got=$(./myls -laO perms_lab | awk 'NR>1 && $NF!="." && $NF!=".." {print $1,$2,$NF}')
[ "$exp" = "$got" ] && echo "VERIFIED: octal + symbolic permission of all $(echo "$exp" | wc -l) entries match stat(1)" || { echo MISMATCH; diff <(echo "$exp") <(echo "$got"); }


$ ./myls -lO perms_lab
total 16
1777 drwxrwxrwt 2 root root 4096 Nov 20 12:27 d_1777
2775 drwxrwsr-x 2 root root 4096 Nov 20 12:27 d_2775
0700 drwx------ 2 root root 4096 Nov 20 12:27 d_700
0755 drwxr-xr-x 2 root root 4096 Nov 20 12:27 d_755
0000 ---------- 1 root root    0 Nov 20 12:27 f_000
1644 -rw-r--r-T 1 root root    0 Nov 20 12:27 f_1644
2755 -rwxr-sr-x 1 root root    0 Nov 20 12:27 f_2755
0400 -r-------- 1 root root    0 Nov 20 12:27 f_400
4644 -rwSr--r-- 1 root root    0 Nov 20 12:27 f_4644
4755 -rwsr-xr-x 1 root root    0 Nov 20 12:27 f_4755
0600 -rw------- 1 root root    0 Nov 20 12:27 f_600
0644 -rw-r--r-- 1 root root    0 Nov 20 12:27 f_644
0755 -rwxr-xr-x 1 root root    0 Nov 20 12:27 f_755
0777 -rwxrwxrwx 1 root root    0 Nov 20 12:27 f_777
7777 -rwsrwsrwt 1 root root    0 Nov 20 12:27 f_7777

VERIFIED: octal + symbolic permission of all 15 entries match stat(1)


**Reading the output** — `f_4755 → -rwsr-xr-x` (setuid + x ⇒ lowercase `s`) · `f_4644 → -rwSr--r--` (setuid, *no* owner
x ⇒ capital `S`) · `f_2755 → -rwxr-sr-x` (setgid) · `f_1644 → -rw-r--r-T` (sticky without other-x ⇒ `T`) ·
`d_1777 → drwxrwxrwt` (sticky directory, like `/tmp`) · `d_2775 → drwxrwsr-x` (setgid directory, used for shared project
folders) · `f_7777 → -rwsrwsrwt` (all special bits).

In [12]:
%%bash
exec 2>&1
echo 'Real system objects (directories shown themselves via -d):'
./myls -ldO /tmp /etc/shadow /usr/bin/passwd /usr/bin/mount /dev/null 2>&1
echo
echo 'Reference:'
for f in /tmp /etc/shadow /usr/bin/passwd /usr/bin/mount /dev/null; do [ -e $f ] && printf '%04d %s %s\n' "$(stat -c %a $f)" "$(stat -c %A $f)" "$f"; done


Real system objects (directories shown themselves via -d):
0666 crw-rw-rw- 1 root root    1, 3 Nov 20 05:56 /dev/null
0640 -rw-r----- 1 root shadow   609 Nov 17 20:52 /etc/shadow
1777 drwxrwxrwt 6 root root    4096 Nov 20 12:27 /tmp
4755 -rwsr-xr-x 1 root root   51584 Aug 19 16:47 /usr/bin/mount
4755 -rwsr-xr-x 1 root root   64152 May 30  2024 /usr/bin/passwd

Reference:
1777 drwxrwxrwt /tmp
0640 -rw-r----- /etc/shadow
4755 -rwsr-xr-x /usr/bin/passwd
4755 -rwsr-xr-x /usr/bin/mount
0666 crw-rw-rw- /dev/null


### 6.2 Exhaustive check — all 4096 permission patterns
Create one file for every mode `0000 … 7777`, then compare `myls`'s octal **and** symbolic output with Python's
`stat.filemode()` — a third, independent implementation.

In [13]:
%%bash
rm -rf allperms && mkdir allperms
python3 - <<'PY'
import os
for i in range(4096):
    p = f"allperms/m{i:04o}"; open(p, "w").close(); os.chmod(p, i)
PY
python3 - <<'PY'
import os, stat, subprocess
out = subprocess.run(["./myls", "-lO", "allperms"], capture_output=True, text=True).stdout.splitlines()[1:]
bad = 0
for line in out:
    octal, sym, *_, name = line.split()
    mode = os.lstat("allperms/" + name).st_mode
    if sym != stat.filemode(mode) or octal != f"{mode & 0o7777:04o}": bad += 1
print(f"files listed: {len(out)}   mismatches vs stat.filemode(): {bad}")
print("RESULT:", "ALL 4096 PERMISSION PATTERNS CORRECT" if bad == 0 and len(out) == 4096 else "FAILED")
print("samples:", *[l.split()[0] + " " + l.split()[1] for l in (out[0], out[0o644], out[0o755], out[0o4755], out[0o2755], out[0o1777], out[0o7777])], sep="\n   ")
PY


files listed: 4096   mismatches vs stat.filemode(): 0
RESULT: ALL 4096 PERMISSION PATTERNS CORRECT
samples:
   0000 ----------
   0644 -rw-r--r--
   0755 -rwxr-xr-x
   4755 -rwsr-xr-x
   2755 -rwxr-sr-x
   1777 -rwxrwxrwt
   7777 -rwsrwsrwt


### 6.3 File-type characters
One object of every type the kernel supports: regular, directory, symlink, FIFO, socket, character and block device.

In [14]:
%%bash
rm -rf types_lab && mkdir types_lab && cd types_lab
touch regular; mkdir dir; ln -s regular symlink; mkfifo fifo
python3 -c "import socket; s=socket.socket(socket.AF_UNIX); s.bind('sock')"
mknod chardev c 1 3 2>/dev/null; mknod blockdev b 7 0 2>/dev/null
cd ..
./myls -lF types_lab


total 4
brw-r--r-- 1 root root 7, 0 Nov 20 12:27 blockdev
crw-r--r-- 1 root root 1, 3 Nov 20 12:27 chardev
drwxr-xr-x 2 root root 4096 Nov 20 12:27 dir/
prw-r--r-- 1 root root    0 Nov 20 12:27 fifo|
-rw-r--r-- 1 root root    0 Nov 20 12:27 regular
srwxr-xr-x 1 root root    0 Nov 20 12:27 sock=
lrwxrwxrwx 1 root root    7 Nov 20 12:27 symlink -> regular


## 7. Error handling and access control
`root` bypasses permission bits (it holds `CAP_DAC_OVERRIDE`), so enforcement is tested as **uid 65534 (`nobody`)** via
`setpriv`. Four directories test the four combinations of read (`r`) / search (`x`) permission on a directory:

| directory | mode | can list names (needs `r`) | can `stat` entries (needs `x`) |
|---|---|---|---|
| `locked` | `000` | no | no |
| `listonly` | `444` | yes | **no** |
| `searchonly` | `111` | no | yes |
| `open` | `755` | yes | yes |

In [15]:
%%bash
exec 2>&1
# fixtures + a world-executable copy of the binary live in /tmp so uid 65534 can reach them
rm -rf /tmp/acl_lab && mkdir -p /tmp/acl_lab/open /tmp/acl_lab/locked /tmp/acl_lab/listonly /tmp/acl_lab/searchonly
install -m 755 myls /tmp/myls_acl
cd /tmp
echo a > acl_lab/open/a; echo a > acl_lab/listonly/a; echo b > acl_lab/listonly/b; echo k > acl_lab/searchonly/known
chmod 000 acl_lab/locked; chmod 444 acl_lab/listonly; chmod 111 acl_lab/searchonly; chmod 755 acl_lab
N="setpriv --reuid=65534 --regid=65534 --clear-groups"
run() { echo "\$ [uid 65534] myls $*"; $N /tmp/myls_acl "$@"; echo "  exit status: $?"; echo; }
run acl_lab/locked
run acl_lab/searchonly
run acl_lab/listonly
run -l acl_lab/listonly
run -R acl_lab
run -l acl_lab/searchonly/known
echo "--- GNU ls: identical diagnostics and exit codes (for comparison) ---"
for d in locked searchonly; do echo "\$ [uid 65534] ls acl_lab/$d"; $N ls acl_lab/$d; echo "  exit status: $?"; done
rm -f /tmp/myls_acl


$ [uid 65534] myls acl_lab/locked
myls: cannot open directory 'acl_lab/locked': Permission denied
  exit status: 2

$ [uid 65534] myls acl_lab/searchonly
myls: cannot open directory 'acl_lab/searchonly': Permission denied
  exit status: 2

$ [uid 65534] myls acl_lab/listonly
a
b
  exit status: 0

$ [uid 65534] myls -l acl_lab/listonly
myls: cannot access 'acl_lab/listonly/b': Permission denied
myls: cannot access 'acl_lab/listonly/a': Permission denied
total 0
?????????? ? ? ? ? ? a
?????????? ? ? ? ? ? b
  exit status: 1

$ [uid 65534] myls -R acl_lab
acl_lab:
listonly
locked
open
searchonly

acl_lab/listonly:
a
b
myls: cannot open directory 'acl_lab/locked': Permission denied

acl_lab/open:
a
myls: cannot open directory 'acl_lab/searchonly': Permission denied
  exit status: 1

$ [uid 65534] myls -l acl_lab/searchonly/known
-rw-r--r-- 1 root root 2 Nov 20 12:27 acl_lab/searchonly/known
  exit status: 0

--- GNU ls: identical diagnostics and exit codes (for comparison) ---
$ [uid 65534

In [16]:
%%bash
exec 2>&1
echo '$ ./myls nosuchdir';                     ./myls nosuchdir;                     echo "  exit status: $?"
echo '$ ./myls demo/README.md nosuch demo/src'; ./myls demo/README.md nosuch demo/src; echo "  exit status: $?"
echo '$ ./myls -Z demo';                       ./myls -Z demo;                       echo "  exit status: $?"


$ ./myls nosuchdir
myls: cannot access 'nosuchdir': No such file or directory
  exit status: 2
$ ./myls demo/README.md nosuch demo/src
myls: cannot access 'nosuch': No such file or directory
demo/README.md

demo/src:
main.c
utils
  exit status: 2
$ ./myls -Z demo
myls: invalid option -- 'Z'
usage: myls [-adlRiShtr1FO] [file ...]
  exit status: 2


### 7.1 Terminal (column) mode
When stdout is a terminal, entries are laid out in columns, filled top-to-bottom like `ls`. Run under a pseudo-terminal 60 columns wide:

In [17]:
%%bash
mkdir -p /tmp/cols && cd /tmp/cols && rm -f * && touch alpha.c beta.c gamma.c delta.h epsilon.h zeta.txt eta.txt theta.md iota.md kappa.py lambda.py mu.go
echo '--- myls (60 cols) ---'; script -qec "stty cols 60; $OLDPWD/myls /tmp/cols" /dev/null </dev/null | tr -d '\r'
echo '--- GNU ls (60 cols) ---'; script -qec "stty cols 60; ls /tmp/cols" /dev/null </dev/null | tr -d '\r'


--- myls (60 cols) ---
alpha.c    epsilon.h  iota.md    mu.go
beta.c     eta.txt    kappa.py   theta.md
delta.h    gamma.c    lambda.py  zeta.txt
--- GNU ls (60 cols) ---
alpha.c  delta.h    eta.txt  iota.md   lambda.py  theta.md
beta.c	 epsilon.h  gamma.c  kappa.py  mu.go	  zeta.txt


## 8. Automated test suite
`run_tests.sh` builds a fixture tree (flat, hidden, nested, empty, 1500-file, 200-character name, symlinks, hard links,
FIFO/socket/devices, 29 files + 8 dirs with distinct permission modes, files from 0 B to 5 MB, timestamps from the
future to 2000 days old), then runs `myls` and GNU `ls` with identical arguments and requires **identical stdout,
stderr and exit status**. Permission tests use `stat(1)` and Python as independent oracles.

In [18]:
%%writefile run_tests.sh
#!/usr/bin/env bash
# run_tests.sh - build fixtures and test ./myls against GNU ls and against stat(1).
# Usage: ./run_tests.sh [path-to-myls]      (exit status = number of failed tests, capped at 100)
export LC_ALL=C TZ=UTC
MYLS=$(readlink -f "${1:-./myls}")
ROOT=${ROOT:-/tmp/myls_lab}
# unprivileged user must be able to execute the binary even if $HOME is not world-traversable
NOBODY_BIN=$(mktemp /tmp/myls_bin.XXXXXX); cp "$MYLS" "$NOBODY_BIN"; chmod 755 "$NOBODY_BIN"
trap 'rm -f "$NOBODY_BIN"' EXIT
pass=0; fail=0

ok()   { pass=$((pass+1)); printf '  PASS  %s\n' "$1"; }
bad()  { fail=$((fail+1)); printf '  FAIL  %s\n' "$1"; }
title(){ printf '\n== %s ==\n' "$1"; }

# check <args...>: run GNU ls and myls with identical args; stdout, stderr and exit code must match.
check() {
  local label exp got expc gotc
  label="myls $(printf '%q ' "$@")"
  exp=$(command ls "$@" 2>&1); expc=$?
  exp=$(printf '%s' "$exp" | sed 's/^ls:/myls:/')
  got=$("$MYLS" "$@" 2>&1); gotc=$?
  if [[ "$exp" == "$got" && $expc -eq $gotc ]]; then
    ok "$label   (exit $gotc)"
  else
    bad "$label   (exit gnu=$expc mine=$gotc)"
    diff <(printf '%s\n' "$exp") <(printf '%s\n' "$got") | head -15 | sed 's/^/        /'
  fi
}

# same as check but executed as an unprivileged user (root bypasses permission bits)
as_nobody() { setpriv --reuid=65534 --regid=65534 --clear-groups "$@"; }
check_nobody() {
  local label exp got expc gotc
  label="[uid 65534] myls $(printf '%q ' "$@")"
  exp=$(as_nobody ls "$@" 2>&1); expc=$?
  exp=$(printf '%s' "$exp" | sed 's/^ls:/myls:/')
  got=$(as_nobody "$NOBODY_BIN" "$@" 2>&1); gotc=$?
  if [[ "$exp" == "$got" && $expc -eq $gotc ]]; then ok "$label   (exit $gotc)"
  else bad "$label   (exit gnu=$expc mine=$gotc)"
       diff <(printf '%s\n' "$exp") <(printf '%s\n' "$got") | head -15 | sed 's/^/        /'; fi
}

# ------------------------------------------------------------------ fixtures
build_fixtures() {
  chmod -R u+rwx "$ROOT" 2>/dev/null; rm -rf "$ROOT"; mkdir -p "$ROOT"; cd "$ROOT" || exit 99
  chmod 755 "$ROOT"
  mkdir -p empty flat hidden nested/a/b/c nested/e links perms sizes times special big longname dirs
  for f in a.txt B.txt c.log 10.txt 9.txt "with space.txt" "naïve.txt" Zeta alpha; do echo "data $f" > "flat/$f"; done
  echo x > hidden/.secret; mkdir hidden/.config; echo y > hidden/.config/app.conf; echo z > hidden/visible.txt
  echo 1 > nested/a/b/c/d.txt; echo 2 > nested/a/x.txt; echo 3 > nested/top.txt
  echo target > links/real.txt; ln -s real.txt links/soft.txt; ln -s ../flat links/dirlink
  ln -s nowhere links/dangling; ln links/real.txt links/hard.txt; ln -s /etc/passwd links/abs
  for m in 000 400 440 444 600 620 640 644 660 664 666 700 711 740 750 755 770 775 777 \
           4755 4644 2755 2644 6755 6644 1755 1644 7777 7000; do
    touch "perms/f_$m"; chmod "$m" "perms/f_$m"; done
  for m in 700 755 1777 2775 4755 3777 000 555; do mkdir "perms/d_$m"; chmod "$m" "perms/d_$m"; done
  for sz in 0 1 1023 1024 1025 1500 10240 1048576 5000000; do truncate -s "$sz" "sizes/s_$sz"; done
  touch -d '30 days ago'   times/recent_30d.txt
  touch -d '300 days ago'  times/older_300d.txt
  touch -d '2000 days ago' times/ancient.txt
  touch -d '1 day ago'     times/yesterday.txt
  touch -d 'tomorrow'      times/future.txt
  touch -d @1600000000     times/tie_b.txt; touch -d @1600000000 times/tie_a.txt
  mkfifo special/pipe
  mknod special/null c 1 3 2>/dev/null; mknod special/loop0 b 7 0 2>/dev/null
  mknod special/tty0 c 4 0 2>/dev/null;  mknod special/pts12 c 136 12 2>/dev/null; mknod special/big c 250 1024 2>/dev/null
  python3 -c "import socket;s=socket.socket(socket.AF_UNIX);s.bind('special/sock')"
  (cd big && seq -f 'file_%04g' 1 1500 | xargs touch)
  touch "longname/$(printf 'x%.0s' $(seq 1 200)).txt"
  mkdir -p dirs/ok/sub dirs/locked/inner dirs/listonly dirs/searchonly
  echo hi > dirs/ok/sub/f; echo hi > dirs/listonly/a; echo hi > dirs/listonly/b; echo hi > dirs/searchonly/known
  chmod 000 dirs/locked; chmod 444 dirs/listonly; chmod 111 dirs/searchonly
  chmod 644 times/* sizes/* 2>/dev/null
}

echo "myls test-suite   $(date -u '+%F %T UTC')"
echo "binary : $MYLS"
echo "gnu ls : $(ls --version | head -1)"
echo "kernel : $(uname -sr)    running as uid $(id -u)"
build_fixtures
echo "fixtures built in $ROOT: $(find "$ROOT" | wc -l) filesystem objects"
# guard: if a fixture is missing, 'both programs fail identically' would be a false PASS
missing=0
for p in empty flat/a.txt hidden/.secret hidden/.config/app.conf nested/a/b/c/d.txt nested/e links/dirlink links/dangling \
         perms/f_7777 sizes/s_5000000 times/future.txt special/pipe special/sock big/file_1500 dirs/locked dirs/listonly; do
  [[ -e $p || -L $p ]] || { echo "  FIXTURE MISSING: $p"; missing=1; }
done
[[ $missing -eq 0 ]] && echo "fixture sanity check: all expected paths exist" || exit 99

# ------------------------------------------------------------------ 1. basic listings
title "1. Basic listing (compared byte-for-byte with GNU ls)"
check flat
check -1 flat
check empty
check -a empty
check hidden
check -a hidden
check -r flat
check -ra flat
check -l flat
check -la flat
check -i flat
check -il flat
check -d flat
check -ld flat hidden perms/d_1777
check -ld .
check -d links/dirlink
check -ld links/dirlink

# ------------------------------------------------------------------ 2. structures
title "2. Directory structures"
check nested
check -R nested
check -R .
check -laR nested
check -R hidden
check -aR hidden
check flat hidden
check flat empty hidden
check flat/a.txt flat/B.txt
check flat/a.txt hidden
check -l flat/a.txt hidden empty
check big
check -l big
check -R big
check longname
check -l longname
check . ..
check ./flat/
check flat//

# ------------------------------------------------------------------ 3. sorting & formats
title "3. Sorting and formatting options"
check -t times
check -tr times
check -lt times
check -S sizes
check -Sr sizes
check -lS sizes
check -lh sizes
check -lhS sizes
check -F flat
check -F links
check -lF links
check -F special
check -l times
check -l -t -r times

# ------------------------------------------------------------------ 4. links and specials
title "4. Symbolic links, hard links, special files"
check -l links
check links
check links/dirlink
check -l links/dirlink
check -l links/dangling links/soft.txt
check -i links
check -l special
check special

# ------------------------------------------------------------------ 5. permission flags
title "5. Permission flags (ls -l mode string vs stat(1), octal via -O)"
perm_dir() {  # compare myls -lO against stat for every entry of a directory
  local d=$1 exp got
  exp=$(cd "$d" && for f in $(command ls -A); do
          printf '%04d %s %s\n' "$(stat -c %a "$f")" "$(stat -c %A "$f")" "$f"; done)
  got=$("$MYLS" -laO "$d" | awk 'NR>1 { n=$NF; for(i=1;i<=NF;i++) if($i=="->"){n=$(i-1)}; if(n!="." && n!="..") print $1, $2, n }')
  if [[ "$exp" == "$got" ]]; then ok "perm bits of $d vs stat: $(printf '%s\n' "$exp" | wc -l) entries identical"
  else bad "perm bits of $d vs stat"; diff <(echo "$exp") <(echo "$got") | head -10 | sed 's/^/        /'; fi
}
perm_dir perms
perm_dir sizes
perm_dir links
perm_dir special

echo "  --- sample of the permission fixtures as listed by myls -lO ---"
"$MYLS" -lO perms | sed 's/^/        /'

# exhaustive: every one of the 4096 permission combinations (rwx x3 + suid/sgid/sticky)
mkdir -p allperms
python3 - <<'PY'
import os
for i in range(4096):
    p = f"allperms/m{i:04o}"
    open(p, "w").close(); os.chmod(p, i)
PY
python3 - "$MYLS" <<'PY'
import os, stat, subprocess, sys
myls = sys.argv[1]
out = subprocess.run([myls, "-lO", "allperms"], capture_output=True, text=True).stdout.splitlines()[1:]
bad = 0
for line in out:
    octal, sym, *_rest, name = line.split()
    mode = os.lstat(f"allperms/{name}").st_mode
    want_sym = stat.filemode(mode)                      # Python's independent implementation
    want_oct = f"{mode & 0o7777:04o}"
    if sym != want_sym or octal != want_oct or name != f"m{want_oct}":
        bad += 1
        if bad <= 5: print("        MISMATCH", line, want_sym, want_oct)
print(f"  {'PASS' if bad == 0 and len(out) == 4096 else 'FAIL'}  exhaustive: {len(out)} files, all 4096 mode patterns "
      f"(0000-7777) match Python stat.filemode(): mismatches = {bad}")
sys.exit(1 if bad or len(out) != 4096 else 0)
PY
[[ $? -eq 0 ]] && pass=$((pass+1)) || fail=$((fail+1))

# file-type characters: - d l c b p s
types=$("$MYLS" -l special links flat/a.txt nested 2>/dev/null | awk 'NF>=9 && $1 ~ /^[-dlcbps]/ {print substr($1,1,1)}' | sort -u | tr -d '\n')
[[ "$types" == *-* && "$types" == *d* && "$types" == *l* && "$types" == *p* && "$types" == *s* ]] \
  && ok "file-type characters seen in output: '$types' (regular, dir, symlink, fifo, socket$( [[ $types == *c* ]] && echo ', char-dev'))" \
  || bad "file-type characters seen: '$types'"

# ------------------------------------------------------------------ 6. errors & access control
title "6. Error handling and permission enforcement"
check nosuchdir
check flat nosuchdir hidden
check -l nosuchfile flat/a.txt
echo "  --- running as unprivileged uid 65534 (root would bypass permission bits) ---"
check_nobody dirs/locked
check_nobody -l dirs/locked
check_nobody dirs/listonly
check_nobody dirs/searchonly
check_nobody -R dirs
check_nobody -l perms/f_000 perms/f_4755
check_nobody -lh sizes
echo "  --- -l inside a list-only (r--) directory: entries cannot be stat()ed ---"
got=$(as_nobody "$NOBODY_BIN" -l dirs/listonly 2>&1 >/dev/null); gotc=$?
exp=$(as_nobody ls -l dirs/listonly 2>&1 >/dev/null | sed 's/^ls:/myls:/')
if [[ "$got" == "$exp" && $gotc -eq 1 ]]; then ok "stderr identical to GNU ls, exit status 1 (minor problem): $(echo "$got" | head -1)"
else bad "list-only -l stderr/exit: exit=$gotc"; fi
check_exit() { "$MYLS" "$@" >/dev/null 2>&1; echo $?; }
[[ $(check_exit -Z flat) -eq 2 ]] && ok "unknown option -Z -> usage message, exit 2" || bad "unknown option exit status"

# ------------------------------------------------------------------ 7. robustness
title "7. Robustness"
n=$("$MYLS" -1 big | wc -l); [[ $n -eq 1500 ]] && ok "1500-entry directory fully read through repeated getdents64 calls ($n lines)" || bad "big dir count $n"
n=$("$MYLS" -1a big | wc -l); [[ $n -eq 1502 ]] && ok "-a adds . and .. ($n lines)" || bad "-a count $n"
out=$("$MYLS" -1 flat | tr '\n' '|'); [[ "$out" == '10.txt|9.txt|B.txt|Zeta|a.txt|alpha|c.log|naïve.txt|with space.txt|' ]] \
  && ok "byte-order (C locale) sort: $out" || bad "sort order: $out"
check -l /dev/null
check /proc/self/fd

# ------------------------------------------------------------------ 8. real system trees
title "8. Real-world trees (thousands of entries, compared with GNU ls)"
for d in /usr/include /etc /usr/bin /dev; do
  [[ -d $d ]] || { echo "  SKIP  $d not present"; continue; }
  case $d in
    /usr/include) check -R "$d"; check -laR "$d" ;;
    /etc)         check -lR "$d"; check -laiR "$d" ;;
    /usr/bin)     check -lS "$d"; check -FR "$d"; check -lhtr "$d" ;;
    /dev)         check -l "$d"; check -la "$d"; check -lh "$d" ;;
  esac
done

echo
echo "================ SUMMARY: $pass passed, $fail failed ================"
chmod -R u+rwx "$ROOT" 2>/dev/null
exit $(( fail > 100 ? 100 : fail ))


Writing run_tests.sh


In [19]:
%%bash
chmod +x run_tests.sh && bash -n run_tests.sh && echo 'run_tests.sh: syntax OK'

run_tests.sh: syntax OK


In [20]:
%%bash
./run_tests.sh ./myls

myls test-suite   2026-11-20 12:27:12 UTC
binary : /home/claude/nb_run/myls
gnu ls : ls (GNU coreutils) 9.4
kernel : Linux 6.18.44-fc-v726    running as uid 0
fixtures built in /tmp/myls_lab: 1610 filesystem objects
fixture sanity check: all expected paths exist

== 1. Basic listing (compared byte-for-byte with GNU ls) ==
  PASS  myls flat    (exit 0)
  PASS  myls -1 flat    (exit 0)
  PASS  myls empty    (exit 0)
  PASS  myls -a empty    (exit 0)
  PASS  myls hidden    (exit 0)
  PASS  myls -a hidden    (exit 0)
  PASS  myls -r flat    (exit 0)
  PASS  myls -ra flat    (exit 0)
  PASS  myls -l flat    (exit 0)
  PASS  myls -la flat    (exit 0)
  PASS  myls -i flat    (exit 0)
  PASS  myls -il flat    (exit 0)
  PASS  myls -d flat    (exit 0)
  PASS  myls -ld flat hidden perms/d_1777    (exit 0)
  PASS  myls -ld .    (exit 0)
  PASS  myls -d links/dirlink    (exit 0)
  PASS  myls -ld links/dirlink    (exit 0)

== 2. Directory structures ==
  PASS  myls nested    (exit 0)
  PASS  myls -

## 9. Memory-safety and mutation testing

**9.1 AddressSanitizer + UndefinedBehaviorSanitizer + LeakSanitizer** — rebuild with sanitizers and re-run the *entire* suite.
Any out-of-bounds access, use-after-free, leak or undefined behaviour aborts with a report.

In [21]:
%%bash
gcc -std=gnu11 -g -O1 -fsanitize=address,undefined -fno-omit-frame-pointer -o myls_asan myls.c && echo 'sanitizer build OK'
ASAN_OPTIONS=detect_leaks=1 ./run_tests.sh ./myls_asan 2>&1 | grep -E 'SUMMARY|FAIL|AddressSanitizer|LeakSanitizer|runtime error' 
echo "sanitizer reports in full run: $(ASAN_OPTIONS=detect_leaks=1 ./run_tests.sh ./myls_asan 2>&1 | grep -cE 'AddressSanitizer|LeakSanitizer|runtime error')"


sanitizer build OK
================ SUMMARY: 91 passed, 0 failed ================
sanitizer reports in full run: 0


**9.2 Mutation testing** — a test-suite that never fails proves nothing. Deliberately inject two bugs into a *copy* of the
program (a swapped sticky-bit letter; an inverted `-S` sort) and confirm the suite catches them.

In [22]:
%%bash
python3 - <<'PY'
s = open('myls.c').read()
a = "s[9] = (m & S_ISVTX) ? ((m & S_IXOTH) ? 't' : 'T')"
b = "r = (a->st.st_size < b->st.st_size) ? 1 : -1;"
assert a in s and b in s
open('/tmp/mut1.c', 'w').write(s.replace(a, "s[9] = (m & S_ISVTX) ? ((m & S_IXOTH) ? 'T' : 't')"))
open('/tmp/mut2.c', 'w').write(s.replace(b, "r = (a->st.st_size < b->st.st_size) ? -1 : 1;"))
PY
for n in 1 2; do
  gcc -o /tmp/myls_mut$n /tmp/mut$n.c
  echo "=== MUTANT $n: $([ $n = 1 ] && echo 'sticky bit letters t/T swapped' || echo '-S sort order inverted') ==="
  ./run_tests.sh /tmp/myls_mut$n 2>&1 | grep -E 'FAIL|SUMMARY' | cut -c1-140
done


=== MUTANT 1: sticky bit letters t/T swapped ===
  FAIL  myls -ld flat hidden perms/d_1777    (exit gnu=0 mine=0)
  FAIL  perm bits of perms vs stat
  FAIL  exhaustive: 4096 files, all 4096 mode patterns (0000-7777) match Python stat.filemode(): mismatches = 2048
  FAIL  myls -l /dev    (exit gnu=0 mine=0)
  FAIL  myls -la /dev    (exit gnu=0 mine=0)
  FAIL  myls -lh /dev    (exit gnu=0 mine=0)
================ SUMMARY: 85 passed, 6 failed ================
=== MUTANT 2: -S sort order inverted ===
  FAIL  myls -S sizes    (exit gnu=0 mine=0)
  FAIL  myls -Sr sizes    (exit gnu=0 mine=0)
  FAIL  myls -lS sizes    (exit gnu=0 mine=0)
  FAIL  myls -lhS sizes    (exit gnu=0 mine=0)
  FAIL  myls -lS /usr/bin    (exit gnu=0 mine=0)
================ SUMMARY: 86 passed, 5 failed ================


**9.3 Stress test on real, messy system trees** — tens of thousands of lines compared with GNU `ls`, plus an informational timing comparison (best-effort, single run, not a rigorous benchmark).

In [23]:
%%bash
for args in "-R /usr/include" "-laR /usr/include" "-lR /etc" "-laiR /usr/lib/python3" "-lhR /usr/share/doc" "-FR /usr/bin" "-lS /usr/bin" "-l /dev"; do
  ls $args > /tmp/g.out 2>&1; ge=$?;  ./myls $args > /tmp/m.out 2>&1; me=$?
  sed -i 's/^ls:/myls:/' /tmp/g.out
  cmp -s /tmp/g.out /tmp/m.out && [ $ge = $me ] && r=IDENTICAL || r=DIFFERENT
  printf '%-10s myls %-26s %6s lines\n' $r "$args" "$(wc -l < /tmp/m.out)"
done
echo; echo '--- timing: myls -laR /usr vs GNU ls -laR /usr ---'
echo -n 'lines: '; ./myls -laR /usr 2>/dev/null | wc -l
time (./myls -laR /usr > /dev/null 2>&1)
time (ls     -laR /usr > /dev/null 2>&1)


IDENTICAL  myls -R /usr/include              8108 lines
IDENTICAL  myls -laR /usr/include           10358 lines
IDENTICAL  myls -lR /etc                     1299 lines
IDENTICAL  myls -laiR /usr/lib/python3       4506 lines
IDENTICAL  myls -lhR /usr/share/doc          6234 lines
IDENTICAL  myls -FR /usr/bin                 1068 lines
IDENTICAL  myls -lS /usr/bin                 1068 lines
IDENTICAL  myls -l /dev                       111 lines

--- timing: myls -laR /usr vs GNU ls -laR /usr ---
lines: 172067



real	0m0.260s
user	0m0.116s
sys	0m0.143s

real	0m0.390s
user	0m0.131s
sys	0m0.258s


## 10. Results

| Check | Result |
|---|---|
| Compiles with `-Wall -Wextra -Wpedantic` | ✅ zero warnings |
| Uses `open` + `getdents64` + `lstat` + `readlink` (no `opendir`/`readdir`) | ✅ verified from imported symbols |
| Automated suite vs GNU `ls` / `stat` / Python | ✅ **91 passed, 0 failed** (1610 fixture objects) |
| All 4096 permission patterns (incl. setuid/setgid/sticky `s S t T`) | ✅ 0 mismatches |
| All 7 file types (`- d l c b p s`) | ✅ |
| Permission enforcement as unprivileged uid (`EACCES`, exit 1 / 2) | ✅ matches GNU `ls` |
| AddressSanitizer + UBSan + LeakSanitizer, full suite | ✅ 91 passed, 0 failed, 0 sanitizer reports |
| Mutation test 1 (sticky letters swapped) | ✅ caught — suite reports 6 failing tests |
| Mutation test 2 (`-S` inverted) | ✅ caught — suite reports 5 failing tests |
| Real trees (`/usr/include`, `/etc`, `/usr/bin`, `/dev`, …) | ✅ byte-identical to GNU `ls` |

**Conclusion** — `myls` reproduces GNU `ls` output for the supported options on every tested structure, decodes all
permission flags correctly, and fails safely (same messages and exit codes) when access is denied.

## 11. Bugs found *by the tests* and fixed

Testing was not a formality — it found real defects (a good talking point in a technical round):

| # | Symptom | Found by | Root cause → fix |
|---|---|---|---|
| 1 | Whole `nested/` test group *passed* while the directory did not exist | test review | `mkdir` without `-p` failed silently; both programs printed the same error. → fixture used `mkdir -p`, and a **fixture-sanity guard** now aborts if any expected path is missing |
| 2 | `-lF` printed `link@ -> target` | diff vs GNU | GNU puts the indicator on the link *target* (`link -> dir/`), found with `stat()` of the target → fixed |
| 3 | `-l file dir` misaligned the file's size column | diff vs GNU | GNU computes column widths over **all** operands, including directories → `print_long` takes the extra entries |
| 4 | `Permission denied` directory printed a `dir:` header first | `-R` as `nobody` | GNU prints the header only after `open()` succeeds → fixed |
| 5 | `/dev` listings misaligned `major, minor` | stress test on `/dev` | GNU pads major and minor in separate columns → separate width tracking |
| 6 | Unreadable symlink (`/proc/1/cwd`): wrong message, exit 0 | stress test on `/proc/1` | must say `cannot read symbolic link` and set exit status 1 → fixed |
| 7 | Plain `ls` on a read-only (`r--`) directory reported errors | `nobody` tests | needless `lstat()` on every name; GNU only stats when `-l -t -S -F` need it → lazy stat |
| 8 | `myls -laR /usr` was ~3× slower than GNU `ls` | timing in §9.3 | `getpwuid`/`getgrgid` (NSS lookups) ran for every entry, twice → cache the last uid/gid → now faster than GNU on this tree (informational only: GNU supports far more features, so it is not a like-for-like benchmark) |

## 12. Limitations (deliberate scope)

- Not implemented: colours (`--color`), `-A -L -n -s -c -u -X`, `--time-style`, ACL/SELinux `+`/`.` markers after the mode.
- Column mode uses one uniform column width (GNU sizes every column separately, so GNU packs more names per row).
- In a long listing an entry whose `lstat()` failed is shown as a simplified `?????????? ? ? ? ? ? name` row (GNU's exact placeholder layout differs); the diagnostic and exit status do match.
- Sorting is **byte order** (equals GNU `ls` under `LC_ALL=C`), not locale collation.
- On a terminal GNU `ls` shell-quotes unusual names (`'with space.txt'`); `myls` prints them raw (piped output is identical).
- Linux-only (`getdents64`, `st_mtim`). Owner/group names use `getpwuid`/`getgrgid`.
- `-O` is an extension (not in POSIX `ls`) added to make permission verification straightforward.

## 13. Viva / interview questions this lab prepares you for

1. **Why `getdents64` and not `readdir`?** `readdir` is a libc wrapper that buffers `getdents64` results; calling the
   syscall shows exactly what the kernel returns (variable-length records: inode, offset, record length, type, name).
2. **`stat` vs `lstat`?** `lstat` describes the symlink itself, `stat` follows it. `ls -l` must use `lstat` or it could never print `l…`.
3. **`s` vs `S`, `t` vs `T`?** Special bit set *and* execute set → lowercase; special bit set *without* execute → uppercase (usually a mistake).
4. **Why does `ls` as root ignore permission bits, and how did you test it?** `CAP_DAC_OVERRIDE`; tests run through `setpriv` as uid 65534.
5. **A directory with `r--` but no `x`: what works?** Names can be listed (`getdents64`), but `lstat` on entries fails with `EACCES`. With `--x` it is the reverse.
6. **Why is the directory link count 2 + number of sub-directories?** `.` inside itself, the entry in its parent, and every child's `..`.
7. **How do you spot hard links?** Same inode number (`-i`), link count > 1; symlinks have their own inode and type `l`.
8. **What does `total` in `ls -l` mean?** Sum of `st_blocks` (512-byte units) converted to 1 KiB — disk usage, not byte size; sparse files can contribute 0.
9. **Why `LC_ALL=C` in the tests?** Locale collation reorders names; byte order is deterministic, so both programs can be diffed.
10. **Complexity?** Reading O(n); sorting O(n log n); memory O(n) per directory; recursion depth = directory depth.

## 14. AI-assistance note (tools: GCC, Claude)

Claude was used as a pair-programming and review aid: to draft the first version of the program and test harness, to
suggest edge cases (setuid/sticky letters, read-only vs search-only directories, device-number alignment, root bypassing
permission checks), and to structure this report. **Nothing is taken on trust**: every claim above is backed by an output
cell produced by GCC-compiled code, the code was validated against GNU `ls`, `stat(1)` and Python's `stat.filemode`, bugs
the tests uncovered are listed in §11, and the suite itself was validated by mutation testing (§9.2).
The student remains responsible for being able to explain every line — see §13.

**To reproduce:** `gcc -std=gnu11 -Wall -Wextra -Wpedantic -O2 -o myls myls.c && chmod +x run_tests.sh && ./run_tests.sh ./myls`
(needs Linux, GCC, GNU coreutils, Python 3, `setpriv`; run as root for the `mknod` device fixtures).